In [0]:
shipping_options_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__shipping_options/"
)

shipping_options_df = (
    spark.read
    .option("multiLine", "false")
    .json(shipping_options_path + "*.jsonl.gz")
)

print("Total rows:", shipping_options_df.count())
print("Total columns:", len(shipping_options_df.columns))

shipping_options_df.printSchema()

### Shipping Options Business Field Groups

The business fields in the `shipping_options` dataset fall into four logical groups:

#### 1. Shipping Cost

The shipping-cost fields capture the monetary components associated with fulfilling the order:

- `shipping_cost__*`
- `additional_shipping_cost_per_unit__*`
- `import_charges__*`
- `shipping_cost_type`

The `shipping_cost__*` fields represent the primary shipping charge, while `additional_shipping_cost_per_unit__*` captures any additional per-unit shipping cost.

The `import_charges__*` fields represent applicable import charges. `shipping_cost_type` provides the context for how the shipping cost is represented.

The monetary fields are currently stored as strings because they originate from the eBay API's nested money structure. This will require normalization in the Silver layer into usable numeric amounts together with their corresponding currencies.

### 2. Delivery Estimate

The delivery-related fields describe the expected delivery window and the assumptions used to calculate it:

- `min_estimated_delivery_date`
- `max_estimated_delivery_date`
- `cut_off_date_used_for_estimate`
- `quantity_used_for_estimate`

`min_estimated_delivery_date` and `max_estimated_delivery_date` define the estimated delivery range, while `cut_off_date_used_for_estimate` records the cutoff information used by the API when generating the estimate.

`quantity_used_for_estimate` indicates the quantity considered when calculating the shipping estimate.

These fields will require further profiling to validate timestamp parsing, completeness, and the relationship between the minimum and maximum delivery dates.

### 3. Carrier / Service

The carrier and service fields describe how the item is expected to be shipped:

- `shipping_carrier_code`
- `shipping_service_code`
- `fulfilled_through`
- `type`

`shipping_carrier_code` identifies the shipping carrier, while `shipping_service_code` identifies the specific shipping service.

`fulfilled_through` provides information about the fulfillment mechanism, and `type` provides the shipping-option type returned by the API.

These fields can later be used to analyze shipping methods, carrier distribution, and fulfillment patterns.

### 4. Metadata / Miscellaneous

The remaining business field is:

- `trademark_symbol`

This appears to be auxiliary metadata rather than a core shipping attribute and should not drive the main shipping-cost or delivery analysis.

### Silver Layer Implication
The shipping-options data should remain structurally flexible in Bronze, preserving the raw API representation for traceability.

In Silver, the nested monetary fields should be normalized into numeric amount + currency representations. Date fields should be converted from raw strings into proper timestamps, while carrier, service, fulfillment, and shipping-type fields can be standardized for analytical use.

The raw values should remain available in Bronze so that any Silver-level normalization can be traced back to the original eBay API response.

In [0]:
# =====================================================
# Shipping options cardinality per listing
# =====================================================

from pyspark.sql import functions as F

shipping_counts = (
    shipping_options_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("shipping_option_count")
    )
)

display(
    shipping_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("shipping_option_count").alias("avg_options"),
        F.expr("percentile_approx(shipping_option_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(shipping_option_count, 0.50)").alias("median"),
        F.expr("percentile_approx(shipping_option_count, 0.75)").alias("p75"),
        F.min("shipping_option_count").alias("min_options"),
        F.max("shipping_option_count").alias("max_options")
    )
)

### Shipping Options Cardinality per Listing

The shipping-options table was grouped by `_dlt_parent_id` to determine how many shipping options are associated with each Item Details listing.

| Metric | Result |
|---|---:|
| Listings with shipping options | 58,963 |
| Average options per listing | 1.2012 |
| P25 | 1 |
| Median | 1 |
| P75 | 1 |
| Minimum options | 1 |
| Maximum options | 4 |

Most listings have exactly **one shipping option**, as both the median and 75th percentile are 1.

The average of approximately **1.20 shipping options per listing** indicates that a smaller subset of listings provides multiple shipping choices.

The maximum observed is **4 shipping options for a single listing**, confirming that the relationship is one-to-many rather than strictly one-to-one.

This supports modeling `shipping_options` as a **child table of Item Details**, with `_dlt_parent_id` acting as the parent listing reference:

```text
Item Details
    │
    │ 1
    │
    └──────────< Shipping Options
                   many (1–4 observed)
```

For the Silver layer, shipping options should therefore remain as separate child records rather than being flattened into repeated shipping columns in the main Item Details table.





In [0]:
# =====================================================
# Validate shipping-options → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

shipping_parent_ids = (
    shipping_options_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    shipping_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    shipping_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Shipping-option parent IDs:", shipping_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

### Shipping Options → Item Details Relationship

The relationship between `shipping_options` and the current Item Details dataset was validated using the shipping-option parent identifier.

| Validation | Result |
|---|---:|
| Shipping-option parent IDs | 58,963 |
| Matched to current Item Details | 58,963 |
| Not found in current Item Details | 0 |

All **58,963 shipping-option parent IDs** successfully matched an existing Item Details record.

There are therefore **0 orphaned shipping-option records** in the current snapshot.

Combined with the cardinality analysis, this confirms that `shipping_options` behaves as a child dataset of Item Details, with each listing having between **1 and 4 observed shipping options**.

```text
Item Details
    │
    │ 1
    │
    └──────────< Shipping Options
                   many
```

The _dlt_parent_id can therefore be used as the parent-listing relationship in the Silver layer without requiring unmatched-record handling.

In [0]:
# =====================================================
# Shipping type distribution
# =====================================================

display(
    shipping_options_df
    .groupBy("shipping_cost_type")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("_dlt_parent_id").alias("listing_count")
    )
    .orderBy(F.desc("record_count"))
)

### Shipping Cost Type Distribution

The `shipping_cost_type` field contains three observed states:

| Shipping Cost Type | Record Count | Listing Count |
|---|---:|---:|
| `FIXED` | 59,286 | 50,412 |
| `NULL` | 6,299 | 3,383 |
| `CALCULATED` | 5,241 | 5,168 |

`FIXED` is the dominant shipping cost type, with **59,286 records across 50,412 listings**.

`CALCULATED` appears in **5,241 records across 5,168 listings**, indicating that calculated shipping is used for a smaller portion of the dataset.

There are **6,299 records with a NULL `shipping_cost_type`**, spanning **3,383 listings**. This should not immediately be treated as a data-quality error because the meaning of a missing shipping-cost type depends on the other shipping fields and the way the eBay API represents shipping options.

The difference between `record_count` and `listing_count` also shows that some listings have multiple shipping-option records of the same type.

For the Silver layer, the raw `shipping_cost_type` should be preserved. `FIXED` and `CALCULATED` can be treated as known categories, while NULL values should remain NULL until their source semantics are established rather than being assigned a default category.

In [0]:
# =====================================================
# Shipping option type distribution
# =====================================================

display(
    shipping_options_df
    .groupBy("type")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("_dlt_parent_id").alias("listing_count")
    )
    .orderBy(F.desc("record_count"))
)

### Shipping Option Type Distribution

The `type` field contains a variety of shipping-service descriptions, with the majority concentrated around standard, economy, and expedited shipping.

| Shipping Option Type | Record Count | Listing Count |
|---|---:|---:|
| `Standard Shipping` | 35,326 | 34,793 |
| `Economy Shipping` | 12,002 | 11,892 |
| `Standard Shipping from outside` | 7,043 | 6,938 |
| `Expedited shipping` | 5,517 | 5,488 |
| `Expedited Shipping` | 5,423 | 5,068 |
| `Economy Shipping from outside` | 2,423 | 2,415 |
| `Expedited Shipping from outside` | 1,971 | 1,970 |
| `One-day Shipping` | 580 | 575 |
| `None` | 297 | 297 |
| `Other Shipping` | 136 | 132 |
| `Freight` | 69 | 69 |
| `Express Shipping from outside` | 21 | 21 |
| `Express Shipping` | 15 | 14 |
| `Express services` | 3 | 3 |

`Standard Shipping` is the dominant shipping option type, followed by `Economy Shipping`.

There are also clear variations in the source values, such as:


- `Expedited shipping`
- `Expedited Shipping`

- `Express Shipping`
- `Express services`

This indicates that the `type` field is a descriptive source field rather than a strictly standardized categorical dimension. Differences in capitalization and wording should therefore be handled carefully during Silver-layer normalization.

The `from outside` variants are also explicitly represented by the source, indicating that the shipping type can contain information about whether the service originates outside the relevant marketplace region.

The `None` and `Other Shipping` values are relatively uncommon and should be preserved as source values rather than automatically converted to NULL.

For the Silver layer, the raw `type` value should be retained for traceability. A standardized shipping-service category can be derived later if the business logic requires grouping values such as standard, economy, expedited, express, freight, and international/outside-marketplace shipping.

In [0]:
# =====================================================
# Shipping cost completeness
# =====================================================

cost_profile = shipping_options_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("shipping_cost__value").isNull(), 1).otherwise(0)
    ).alias("shipping_cost_value_null"),

    F.sum(
        F.when(F.col("shipping_cost__currency").isNull(), 1).otherwise(0)
    ).alias("shipping_cost_currency_null"),

    F.sum(
        F.when(
            F.col("shipping_cost__value").isNotNull() &
            (F.trim(F.col("shipping_cost__value")) == ""),
            1
        ).otherwise(0)
    ).alias("shipping_cost_value_blank"),

    F.sum(
        F.when(F.col("import_charges__value").isNull(), 1).otherwise(0)
    ).alias("import_charges_value_null"),

    F.sum(
        F.when(F.col("additional_shipping_cost_per_unit__value").isNull(), 1)
         .otherwise(0)
    ).alias("additional_cost_value_null")
)

display(cost_profile)

### Shipping Cost Completeness

A completeness check was performed on the main shipping-cost fields across all **70,826 shipping-option records**.

| Field | Null Count |
|---|---:|
| `shipping_cost__value` | 0 |
| `shipping_cost__currency` | 0 |
| `shipping_cost__value` blank | 0 |
| `import_charges__value` | 69,763 |
| `additional_shipping_cost_per_unit__value` | 12,275 |

The primary `shipping_cost__value` and `shipping_cost__currency` fields are **100% populated**, with no null or blank shipping-cost values observed.

This means every shipping-option record has a primary shipping cost and its associated currency in the current snapshot.

In contrast, `import_charges__value` is populated for only **1,063 records** and is NULL for **69,763 records**. This indicates that import charges are generally optional rather than a universally populated shipping attribute.

Similarly, `additional_shipping_cost_per_unit__value` is NULL for **12,275 records**, meaning this field is optional and only applies to a subset of shipping options.

The missing values in `import_charges__value` and `additional_shipping_cost_per_unit__value` should therefore **not automatically be treated as data-quality errors**. Their optional nature is consistent with these fields representing additional charges that may not apply to every shipping option.

For the Silver layer:

```text
shipping_cost__value
    → required numeric shipping amount

shipping_cost__currency
    → required currency

import_charges__value
    → nullable numeric amount

additional_shipping_cost_per_unit__value
    → nullable numeric amount
```

The primary shipping cost can be treated as a reliable analytical field, while the additional charge fields should remain nullable when the source does not provide a value.

In [0]:
# =====================================================
# Delivery estimate completeness
# =====================================================

delivery_profile = shipping_options_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("min_estimated_delivery_date").isNull(), 1)
         .otherwise(0)
    ).alias("min_delivery_date_null"),

    F.sum(
        F.when(F.col("max_estimated_delivery_date").isNull(), 1)
         .otherwise(0)
    ).alias("max_delivery_date_null"),

    F.sum(
        F.when(F.col("cut_off_date_used_for_estimate").isNull(), 1)
         .otherwise(0)
    ).alias("cutoff_date_null"),

    F.sum(
        F.when(F.col("shipping_carrier_code").isNull(), 1)
         .otherwise(0)
    ).alias("carrier_null"),

    F.sum(
        F.when(F.col("shipping_service_code").isNull(), 1)
         .otherwise(0)
    ).alias("service_null")
)

display(delivery_profile)

### Delivery Estimate Completeness

A completeness check was performed across the delivery-estimate and shipping-service fields for all **70,826 shipping-option records**.

| Field | Null Count |
|---|---:|
| `min_estimated_delivery_date` | 75 |
| `max_estimated_delivery_date` | 75 |
| `cut_off_date_used_for_estimate` | 67,416 |
| `shipping_carrier_code` | 41,542 |
| `shipping_service_code` | 0 |

The minimum and maximum estimated delivery dates are almost completely populated, with only **75 records** missing each field. This indicates that delivery estimates are available for virtually all shipping options where an estimate is provided.

`cut_off_date_used_for_estimate` is highly sparse, with **67,416 NULL values**. This appears to be an optional source field rather than a required component of every delivery estimate.

`shipping_carrier_code` is also frequently missing, with **41,542 NULL values**. Therefore, carrier information is not consistently available across shipping options and should be treated as optional rather than assuming that every shipping service has a carrier code.

In contrast, `shipping_service_code` has **0 NULL values**, making it the most consistently populated field among the service-related attributes.

For the Silver layer:

```text
min_estimated_delivery_date
    → nullable timestamp

max_estimated_delivery_date
    → nullable timestamp

cut_off_date_used_for_estimate
    → nullable timestamp

shipping_carrier_code
    → nullable categorical field

shipping_service_code
    → required source field
```
The sparse fields should remain nullable rather than being filled with artificial defaults. Further profiling should focus on validating the delivery-date relationships and understanding when carrier and cutoff information is absent.

In [0]:
# =====================================================
# Validate delivery date ordering
# =====================================================

delivery_dates = (
    shipping_options_df
    .select(
        F.to_timestamp("min_estimated_delivery_date").alias("min_delivery"),
        F.to_timestamp("max_estimated_delivery_date").alias("max_delivery")
    )
    .filter(
        F.col("min_delivery").isNotNull() &
        F.col("max_delivery").isNotNull()
    )
)

invalid_delivery_order = delivery_dates.filter(
    F.col("min_delivery") > F.col("max_delivery")
).count()

print("Invalid delivery date ranges:", invalid_delivery_order)

### Delivery Date Ordering

The relationship between the minimum and maximum estimated delivery dates was validated after converting both fields to timestamps.

Among records where both delivery dates are available, **0 invalid date ranges** were found.

This means:

```text
min_estimated_delivery_date <= max_estimated_delivery_date
```
for every shipping option with both dates populated.

There are therefore no cases where the minimum estimated delivery date occurs after the maximum estimated delivery date.

Combined with the completeness check, the delivery-date fields can be safely handled in Silver as nullable timestamps:

- `min_estimated_delivery_date` → nullable timestamp
- `max_estimated_delivery_date` → nullable timestamp

No date-order correction or invalid-range handling is required for the current snapshot.

In [0]:
# =====================================================
# Blank / whitespace-only string check
# =====================================================

string_columns = [
    field.name
    for field in shipping_options_df.schema.fields
    if field.dataType.simpleString() == "string"
]

blank_profile = shipping_options_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in string_columns
])

blank_counts = blank_profile.first().asDict()

blank_data = [
    (column, int(blank_counts[column]))
    for column in string_columns
    if blank_counts[column] > 0
]

if blank_data:
    blank_df = spark.createDataFrame(
        blank_data,
        ["column_name", "blank_count"]
    )
    display(blank_df.orderBy(F.desc("blank_count")))
else:
    print("No empty or whitespace-only string values found.")

### Shipping Options Blank / Whitespace Check

A check was performed across the string fields in the `shipping_options` dataset to identify empty strings and values containing only whitespace.

The result was:

```text
No empty or whitespace-only string values found.
```

Therefore, none of the string fields currently contain non-null values that become empty after trimming whitespace.

This means the shipping-options dataset does not require a separate cleaning step for blank or whitespace-only string values at this stage.

The remaining missing-value handling can focus on actual `NULL` values, particularly in optional fields such as `import_charges__value`, `additional_shipping_cost_per_unit__value`, `cut_off_date_used_for_estimate`, and `shipping_carrier_code`.